# requests

In [1]:
import requests

Основной `GET` запрос для API:
- `timeout` - чтобы не висеть вечно
- `headers` - заголовки
- `params` - query-параметры
- `response.raise_for_status()` - requests не выбрасывает исключение на коды 4xx/5xx автоматически

In [ ]:
headers = {
    "Authorization": "Bearer YOUR_TOKEN",
    "Accept": "application/json",
}
params = {"q": "python requests", "sort": "stars", "order": "desc"}
response = requests.get("https://api.github.com/search/repositories", params=params, headers=headers, timeout=10)
response.raise_for_status()

print(response.url)
# https://api.github.com/search/repositories?q=python+requests&sort=stars&order=desc

**Пагинация — если данных много**

Паттерн через offset / limit:

In [ ]:
all_data = []
offset = 0
limit = 100

while True:
    params = {"limit": limit, "offset": offset}
    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    chunk = response.json()
    if not chunk:          # пустой ответ — данных больше нет
        break

    all_data.extend(chunk)
    offset += limit

Паттерн через page / per_page:

In [ ]:
all_data = []
page = 1

while True:
    params = {"page": page, "per_page": 100}
    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()

    chunk = response.json()
    if not chunk:
        break

    all_data.extend(chunk)
    page += 1

*Если API использует Basic Auth, то вместо Authorization можно так: `response = requests.get(url, auth=("username", "password"))`*

**Как достать данные:**

| Что нужно | Как достать |
|---|---|
| **JSON** (почти всегда для API) | `response.json()` → `dict` / `list` |
| **Сырой текст** | `response.text` |
| **Байты** (файлы, картинки) | `response.content` |

Внешний вид основных методов:

In [ ]:
response = requests.get("https://api.github.com/events")
print(response.status_code)  # 200
requests.post("https://httpbin.org/post", data={"key": "value"})
requests.put("https://httpbin.org/put", data={"key": "value"})
requests.delete("https://httpbin.org/delete")

**Бонус**

Если JSON вложенный (а он почти всегда вложенный в реальных API), используем `pd.json_normalize()` — он «расплющивает» структуру в таблицу

In [ ]:
import pandas as pd

df = pd.DataFrame(all_data)
# or
df = pd.json_normalize(all_data)

и общий паттерн:

In [ ]:
import requests
import pandas as pd

def fetch_all_pages(url, headers, limit=100):
    all_data = []
    offset = 0

    while True:
        params = {"limit": limit, "offset": offset}
        try:
            response = requests.get(
                url,
                headers=headers,
                params=params,
                timeout=30,
            )
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            break

        chunk = response.json()
        if not chunk:
            break

        all_data.extend(chunk)
        offset += limit

    return pd.DataFrame(all_data)


# Использование
headers = {"Authorization": "Bearer YOUR_TOKEN"}
df = fetch_all_pages("https://api.example.com/data", headers=headers)
print(df.shape)